# Development of the Stacked Hybrid SARIMAX-LSTM Models with Rolling Validation

Develops the three stacked hybrid models and evaluates them with **exactly the same expanding-window rolling validation** as the standalone models:
- 8 rounds, validating 2015–2022
- December origin, forecasting the 12 monthly values of the next year
- the same RMSE, MAE and nRMSE outputs per round and grid point

2023–2024 is held out for final testing (`Final_Testing_Hybrid.ipynb`).

## The three stacked hybrid models

| Stack | SARIMAX component | LSTM component | Combination |
|---|---|---|---|
| **1. Hybrid SARIMAX-LSTM** | forecasts GHI | forecasts GHI, MSE loss | time-varying weighted average |
| **2. Hybrid + Ineichen-Perez CSI** | forecasts the clear-sky index (CSI) | forecasts CSI, MSE loss | time-varying weighted average |
| **3. Hybrid + CSI + Penalized Loss** | forecasts CSI (same as stack 2) | forecasts CSI, **penalized loss** | time-varying weighted average |

In stacks 2 and 3, GHI is reconstructed as **GHI = CSI × GHI_clear**, where GHI_clear is the Ineichen-Perez clear-sky GHI of the target month. GHI_clear is deterministic, so it is known in advance for the forecast year.

**Parallel ensemble, independent component combination.** SARIMAX and LSTM are fitted independently on the same series and forecast the full year. Their forecasts are combined only at the output:

`GHI_hybrid = w_S · GHI_SARIMAX + (1 − w_S) · GHI_LSTM`

**Time-varying weighted average (TVWA).** The weight w_S is recomputed **for every grid cell and every forecast year**, using the components' past out-of-sample errors with exponential discounting (Bates & Granger, 1969; Stock & Watson, 2004):

`D_c = Σ_{y < Y} δ^(Y−1−y) · Σ_months (GHI_c − GHI_obs)²`,   `w_S = D_LSTM / (D_SARIMAX + D_LSTM)`

With δ = 0.8, the most recent year counts fully, the year before counts 0.8, and so on. The weights therefore follow whichever component has been more accurate recently. Only forecast years before Y are used, so the weights never see the year being forecast.

**Penalized loss (stack 3).** The LSTM is trained on CSI with

`L = MSE + λ_v · mean[ReLU(ĉ − CSI_max)² + ReLU(−ĉ)²] + λ_e · mean[ReLU(|ĉ − c| − τ)²] − λ_r · mean[exp(−(ĉ − c)² / 2τ²) · 1(0 ≤ ĉ ≤ CSI_max)]`

| Term | Role | Default |
|---|---|---|
| MSE | base accuracy | 1 |
| clear-sky violation penalty | loss for forecasts above the clear-sky threshold (CSI_max = 1, i.e. GHI > GHI_clear) or below 0 | λ_v = 10 |
| inaccuracy penalty | extra loss for errors larger than the tolerance τ = 0.05 CSI | λ_e = 1 |
| accuracy reward | bounded reward (0 to λ_r) for forecasts close to the observation **and** inside the physical bounds | λ_r = 0.005 |

The reward is bounded, so it cannot outweigh the penalties. The violation penalty is zero for physically valid forecasts.

## Rolling rounds

| Round | Training | Forecast origin | Validation year |
|---|---|---|---|
| warm-up | 2005–2013 | Dec 2013 | 2014 (**not scored**; only provides the first round's TVWA weights) |
| 1 | 2005–2014 | Dec 2014 | 2015 |
| … | … | … | … |
| 8 | 2005–2021 | Dec 2021 | 2022 |

SARIMAX orders are selected once, on the first training window (2005–2013), separately for GHI and CSI. The parameters are then re-estimated every round. The LSTMs are retrained from scratch every round with the same architecture and settings as the standalone LSTM.

## Outputs (`outputs/hybrid_models/`)
- `rolling_metrics/<model>_rolling_metrics_by_grid.csv`: **RMSE, MAE and nRMSE per round and grid point** (same format as the standalone models), plus clear-sky violations
- `rolling_metrics/<model>_rolling_metrics_by_round.csv`: per round, all grids pooled
- `rolling_metrics/rolling_metrics_summary.csv`: hybrids, their components and the standalone models (if run) over all rounds
- `rolling_predictions/`: every forecast
- `components/`: raw component forecasts, which are reused by the final-test notebook
- `models/`: SARIMAX orders, TVWA weights per grid and year, LSTM weights

## Setup
```
%pip install statsmodels scikit-learn torch joblib pvlib
```
Run from the `Philippine Spatial Map` folder, with the cells in order. The first run also computes the Ineichen-Perez clear-sky table, which takes about 2 minutes and is cached in `data/processed/`. Total runtime is about 45–60 minutes on a multi-core CPU, mostly from training 3 LSTM variants × 9 rounds.

In [1]:
"""
hybrid_models_rolling.py

Purpose:
    Stacked hybrid SARIMAX-LSTM models (plain, + Ineichen-Perez CSI,
    + penalized loss) with expanding-window rolling validation.

Input:
    data/processed/nasa_power_validated.csv
    data/raw/elevation/grid_elevation.csv

Output:
    outputs/hybrid_models/
"""

STAGE = "rolling"

from pathlib import Path


# ============================================================
# 1. USER SETTINGS
# ============================================================

# ------------------------------------------------------------
# INPUT FILES
# ------------------------------------------------------------

DATA_FILE = Path(
    "data/processed/nasa_power_validated.csv"
)

ELEVATION_FILE = Path(
    "data/raw/elevation/grid_elevation.csv"
)

# Ineichen-Perez clear-sky table (computed on first run)
CLEARSKY_FILE = Path(
    "data/processed/clearsky_ineichen_monthly.csv"
)

# Standalone predictions, only used for the comparison table
STANDALONE_PREDICTION_DIR = Path(
    "outputs/standalone_models/rolling_predictions"
)


# ------------------------------------------------------------
# OUTPUT
# ------------------------------------------------------------

OUTPUT_DIR = Path(
    "outputs/hybrid_models"
)

COMPONENT_DIR = OUTPUT_DIR / "components"

PREDICTION_DIR = OUTPUT_DIR / "rolling_predictions"

METRICS_DIR = OUTPUT_DIR / "rolling_metrics"

MODEL_DIR = OUTPUT_DIR / "models"

# Written here, read by the final-test notebook
SARIMAX_ORDERS_FILE = MODEL_DIR / "sarimax_component_orders.csv"


# ------------------------------------------------------------
# PERIOD
# ------------------------------------------------------------
# 2023-2024 is reserved for final testing and removed on load.

DATA_START = "2005-01-01"

DATA_END = "2022-12-01"

# Scored validation years (same as the standalone models)
EVAL_YEARS = list(range(2015, 2023))

# Extra first forecast year: not scored, only provides the
# component errors needed for the first round's TVWA weights
WARMUP_YEARS = [2014]


# ------------------------------------------------------------
# VARIABLES
# ------------------------------------------------------------

GHI_COL = "ghi_kwh_m2_day"

CLOUD_COL = "cloud_amt_pct"

RH_COL = "rh2m_pct"

# Ineichen-Perez clear-sky GHI and clear-sky index (added in section 1)
CLEAR_COL = "ghi_clear_kwh_m2_day"

CSI_COL = "csi"

VARIABLES = [
    GHI_COL,
    CLOUD_COL,
    RH_COL
]

EXOG_COLS = [
    CLOUD_COL,
    RH_COL
]

STATIC_COLS = [
    "latitude",
    "longitude",
    "elevation_m"
]

CALENDAR_COLS = [
    "month_sin",
    "month_cos"
]


# ------------------------------------------------------------
# FORECAST SETTINGS (same as the standalone models)
# ------------------------------------------------------------

HORIZON = 12

WINDOW = 12

SEASONAL_PERIOD = 12


# ------------------------------------------------------------
# PHYSICAL THRESHOLD
# ------------------------------------------------------------
# Monthly-mean GHI cannot exceed the clear-sky GHI: CSI <= 1

CSI_MAX = 1.0


# ------------------------------------------------------------
# REPRODUCIBILITY
# ------------------------------------------------------------

SEED = 42


# ============================================================
# 2. IMPORTS AND OUTPUT DIRECTORIES
# ============================================================

from importlib import metadata
import platform
import random
import warnings

import numpy as np
import pandas as pd

for directory in [COMPONENT_DIR, PREDICTION_DIR, METRICS_DIR, MODEL_DIR]:

    directory.mkdir(
        parents=True,
        exist_ok=True
    )


# ============================================================
# 3. REPRODUCIBILITY
# ============================================================

def set_seed(seed=SEED):
    """Fix the Python and NumPy random seeds."""

    random.seed(seed)

    np.random.seed(seed)


set_seed()


def library_versions():
    """Versions of every library used, for the paper's reproducibility section."""

    versions = {"python": platform.python_version()}

    for package in ["numpy", "pandas", "statsmodels", "scikit-learn", "torch", "pvlib", "joblib"]:

        try:
            versions[package] = metadata.version(package)
        except metadata.PackageNotFoundError:
            versions[package] = "not installed"

    return versions


VERSIONS = library_versions()

(OUTPUT_DIR / "environment.txt").write_text(
    "\n".join(f"{name}=={version}" for name, version in VERSIONS.items()),
    encoding="utf-8"
)

print("Library versions:")

for name, version in VERSIONS.items():
    print(f"  {name:<13} {version}")


# ============================================================
# 4. LOAD DATA
# ============================================================

print("\nLoading data...")

data = pd.read_csv(
    DATA_FILE,
    parse_dates=["date"]
)

elevation = pd.read_csv(
    ELEVATION_FILE
)[["grid_number", "elevation_m"]]

data = (
    data.merge(elevation, on="grid_number", how="left")
    .sort_values(["grid_number", "date"])
    .reset_index(drop=True)
)

n_before = len(data)

data = data[
    (data["date"] >= DATA_START)
    & (data["date"] <= DATA_END)
].reset_index(drop=True)

print(f"Records after {DATA_END[:7]} removed: {n_before - len(data)}")


# Fill any values set to NaN during validation
n_missing = int(data[VARIABLES].isna().sum().sum())

if n_missing:

    print(f"Interpolating {n_missing} missing values within each grid")

    data[VARIABLES] = (
        data.groupby("grid_number")[VARIABLES]
        .transform(lambda s: s.interpolate(limit_direction="both"))
    )


data["month_sin"] = np.sin(2 * np.pi * data["month"] / 12)

data["month_cos"] = np.cos(2 * np.pi * data["month"] / 12)


GRID_INFO = (
    data[["grid_number", "grid_id", "latitude", "longitude", "elevation_m"]]
    .drop_duplicates("grid_number")
    .reset_index(drop=True)
)

N_GRIDS = len(GRID_INFO)

print(
    f"Grids: {N_GRIDS}   "
    f"Months: {data['date'].min():%Y-%m} to {data['date'].max():%Y-%m}   "
    f"Records: {len(data)}"
)


# ============================================================
# 5. FORECAST ROUNDS
# ============================================================
# COMPONENT_ROUNDS: every year the components forecast
# (warm-up years + scored years)

COMPONENT_ROUNDS = pd.DataFrame([
    {
        "year":           year,
        "train_end_date": pd.Timestamp(f"{year - 1}-12-01"),
        "forecast_start": pd.Timestamp(f"{year}-01-01"),
        "forecast_end":   pd.Timestamp(f"{year}-12-01")
    }
    for year in WARMUP_YEARS + EVAL_YEARS
])

print("\nComponent forecast years:")

print(
    COMPONENT_ROUNDS
    .assign(
        training=lambda r: DATA_START[:4] + "-" + (r["year"] - 1).astype(str),
        scored=lambda r: r["year"].isin(EVAL_YEARS)
    )[["year", "training", "scored"]]
    .to_string(index=False)
)


# ============================================================
# 6. EVALUATION ROUNDS AND METRICS (same as standalone models)
# ============================================================

ROUNDS = pd.DataFrame([
    {
        "round":           number,
        "validation_year": year,
        "train_start":     DATA_START[:7],
        "train_end":       f"{year - 1}-12"
    }
    for number, year in enumerate(EVAL_YEARS, start=1)
])

ROUND_COLS = ["round", "validation_year", "train_start", "train_end"]

GRID_COLS = ["grid_number", "grid_id", "latitude", "longitude"]

YEAR_COL = "validation_year"


def summarise_errors(frame, keys):
    """
    RMSE, MAE (kWh/m²/day) and nRMSE (%) per group, plus the
    number of clear-sky violations (forecast GHI > clear-sky GHI).
    """

    error = frame["predicted"] - frame["actual"]

    summary = (
        frame.assign(
            squared_error=error ** 2,
            absolute_error=error.abs(),
            violation=frame["predicted"] > frame[CLEAR_COL]
        )
        .groupby(keys, sort=True)
        .agg(
            RMSE=("squared_error", "mean"),
            MAE=("absolute_error", "mean"),
            mean_actual=("actual", "mean"),
            mean_predicted=("predicted", "mean"),
            clear_sky_violations=("violation", "sum"),
            n_months=("actual", "size")
        )
        .reset_index()
    )

    summary["RMSE"] = np.sqrt(summary["RMSE"])

    summary["nRMSE_pct"] = summary["RMSE"] / summary["mean_actual"] * 100

    summary["mean_bias_pct"] = (
        (summary["mean_predicted"] - summary["mean_actual"])
        / summary["mean_actual"] * 100
    )

    return summary[
        keys
        + ["RMSE", "MAE", "nRMSE_pct", "mean_actual", "mean_predicted",
           "mean_bias_pct", "clear_sky_violations", "n_months"]
    ]


def attach_actuals(predictions, label):
    """
    Add round, observed GHI, clear-sky GHI and grid information
    to forecasts with columns: round, grid_number, target_date, predicted.
    """

    frame = (
        predictions[["round", "grid_number", "target_date", "predicted"]]
        .merge(ROUNDS[ROUND_COLS], on="round", how="left")
        .merge(
            data[["grid_number", "date", GHI_COL, CLEAR_COL]]
            .rename(columns={"date": "target_date", GHI_COL: "actual"}),
            on=["grid_number", "target_date"],
            how="left"
        )
        .merge(GRID_INFO[GRID_COLS], on="grid_number", how="left")
    )

    frame["model"] = label

    frame["horizon_months"] = frame["target_date"].dt.month

    frame["error"] = frame["predicted"] - frame["actual"]

    expected_rows = len(ROUNDS) * N_GRIDS * HORIZON

    wrong_year = frame["target_date"].dt.year != frame[YEAR_COL]

    if (
        len(frame) != expected_rows
        or frame[["actual", "predicted"]].isna().any().any()
        or wrong_year.any()
    ):

        raise ValueError(
            f"{label}: {len(frame)} forecasts (expected {expected_rows}), "
            f"{frame['predicted'].isna().sum()} missing predictions, "
            f"{wrong_year.sum()} outside their year"
        )

    return frame[
        ["model"] + ROUND_COLS + GRID_COLS
        + ["target_date", "horizon_months", "actual", "predicted", "error", CLEAR_COL]
    ].sort_values(["round", "grid_number", "target_date"]).reset_index(drop=True)


def save_results(predictions, name, label):
    """
    Save forecasts and rolling metrics in the standalone format:
        rolling_predictions/<name>_rolling_predictions.csv
        rolling_metrics/<name>_rolling_metrics_by_grid.csv
        rolling_metrics/<name>_rolling_metrics_by_round.csv
    """

    frame = attach_actuals(predictions, label)

    by_grid = summarise_errors(frame, ["model"] + ROUND_COLS + GRID_COLS)

    by_round = summarise_errors(frame, ["model"] + ROUND_COLS)

    frame.to_csv(PREDICTION_DIR / f"{name}_rolling_predictions.csv", index=False)

    by_grid.to_csv(METRICS_DIR / f"{name}_rolling_metrics_by_grid.csv", index=False)

    by_round.to_csv(METRICS_DIR / f"{name}_rolling_metrics_by_round.csv", index=False)

    overall = summarise_errors(frame, ["model"]).iloc[0]

    print(
        f"{label:<34} RMSE {overall['RMSE']:.4f}   MAE {overall['MAE']:.4f}   "
        f"nRMSE {overall['nRMSE_pct']:.2f}%   "
        f"clear-sky violations {int(overall['clear_sky_violations'])}"
    )

    return frame

Library versions:
  python        3.14.7
  numpy         2.5.3
  pandas        3.0.6
  statsmodels   0.15.0
  scikit-learn  1.9.1
  torch         2.14.0
  pvlib         0.16.1
  joblib        1.6.0

Loading data...
Records after 2022-12 removed: 2232
Grids: 93   Months: 2005-01 to 2022-12   Records: 20088

Component forecast years:
 year  training  scored
 2014 2005-2013   False
 2015 2005-2014    True
 2016 2005-2015    True
 2017 2005-2016    True
 2018 2005-2017    True
 2019 2005-2018    True
 2020 2005-2019    True
 2021 2005-2020    True
 2022 2005-2021    True


## 1. Ineichen-Perez clear-sky GHI and clear-sky index (CSI)

The **Ineichen-Perez clear-sky model** (Ineichen & Perez, 2002), as implemented in pvlib, gives the GHI expected under a cloudless sky at each grid point's NASA POWER query coordinate.

- **Inputs:**
  - solar position for each hour
  - site altitude: SRTM elevation
  - Linke turbidity: the monthly SoDa climatology bundled with pvlib
- **Aggregation:** hourly values (sampled at the middle of each hour, local time Asia/Manila) are summed to daily clear-sky irradiation, then averaged by month (kWh/m²/day). This matches the NASA POWER monthly GHI.
- **Clear-sky index:** `CSI = GHI / GHI_clear`. It removes the deterministic, astronomical part of the seasonal cycle and leaves the cloud- and aerosol-driven part that the models must learn.
- **Physical threshold:** monthly-mean GHI should not exceed the clear-sky GHI, so **CSI ≤ 1**.

The table covers 2005–2026, so it can also be used for the 2026 forecast. It is computed once (about 2 minutes) and cached in `data/processed/clearsky_ineichen_monthly.csv`.

In [2]:
# ============================================================
# 1. INEICHEN-PEREZ CLEAR-SKY GHI AND CLEAR-SKY INDEX
# ============================================================

import pvlib
from pvlib.location import Location


CLEARSKY_START = "2005-01-01"

CLEARSKY_END = "2026-12-31"

TIMEZONE = "Asia/Manila"


def monthly_clearsky_ghi(latitude, longitude, altitude, times):
    """
    Monthly mean of daily Ineichen-Perez clear-sky GHI (kWh/m²/day).
    Hourly clear-sky GHI (W/m²) at mid-hour is summed per day
    (Wh/m²), converted to kWh/m² and averaged per month.
    """

    site = Location(
        latitude,
        longitude,
        tz=TIMEZONE,
        altitude=altitude
    )

    # Linke turbidity is looked up from pvlib's monthly climatology
    clearsky = site.get_clearsky(times, model="ineichen")

    daily = clearsky["ghi"].resample("D").sum() / 1000

    monthly = daily.resample("MS").mean()

    return pd.Series(
        monthly.to_numpy(),
        index=monthly.index.tz_localize(None)
    )


if CLEARSKY_FILE.exists():

    print(f"Loading cached clear-sky table: {CLEARSKY_FILE}")

    clearsky = pd.read_csv(CLEARSKY_FILE, parse_dates=["date"])

else:

    print("Computing Ineichen-Perez clear-sky GHI for every grid (about 2 minutes)...")

    times = pd.date_range(
        f"{CLEARSKY_START} 00:30",
        f"{CLEARSKY_END} 23:30",
        freq="1h",
        tz=TIMEZONE
    )

    tables = []

    for site in GRID_INFO.itertuples(index=False):

        monthly = monthly_clearsky_ghi(
            site.latitude,
            site.longitude,
            site.elevation_m,
            times
        )

        tables.append(pd.DataFrame({
            "grid_number": site.grid_number,
            "date":        monthly.index,
            CLEAR_COL:     monthly.to_numpy()
        }))

    clearsky = pd.concat(tables, ignore_index=True)

    CLEARSKY_FILE.parent.mkdir(parents=True, exist_ok=True)

    clearsky.to_csv(CLEARSKY_FILE, index=False)

    print(f"Saved: {CLEARSKY_FILE}")


data = data.drop(columns=[CLEAR_COL, CSI_COL], errors="ignore").merge(
    clearsky[["grid_number", "date", CLEAR_COL]],
    on=["grid_number", "date"],
    how="left"
)

if data[CLEAR_COL].isna().any():

    raise ValueError("Clear-sky GHI missing for some grid-months")

data[CSI_COL] = data[GHI_COL] / data[CLEAR_COL]


print(
    f"\nClear-sky GHI: {data[CLEAR_COL].min():.3f} - {data[CLEAR_COL].max():.3f} kWh/m²/day "
    f"(mean {data[CLEAR_COL].mean():.3f})"
)

print(
    f"CSI: {data[CSI_COL].min():.4f} - {data[CSI_COL].max():.4f} "
    f"(mean {data[CSI_COL].mean():.4f})"
)

print(
    f"Observed months above the clear-sky threshold (CSI > {CSI_MAX}): "
    f"{(data[CSI_COL] > CSI_MAX).sum()} of {len(data)}"
)

Computing Ineichen-Perez clear-sky GHI for every grid (about 2 minutes)...
Saved: data\processed\clearsky_ineichen_monthly.csv

Clear-sky GHI: 4.705 - 8.165 kWh/m²/day (mean 6.739)
CSI: 0.3910 - 1.0176 (mean 0.7523)
Observed months above the clear-sky threshold (CSI > 1.0): 14 of 20088


## 2. SARIMAX components (GHI and CSI)

This uses the same SARIMAX set-up as the standalone model, fitted per grid cell for two target series:
- `sarimax_ghi` forecasts GHI (component of stack 1)
- `sarimax_csi` forecasts CSI, which is converted to GHI with the clear-sky GHI (component of stacks 2 and 3)

**Settings:**
- **Exogenous regressors:** cloud amount and RH lagged 12 months.
- **Seasonal differencing:** D = 1, s = 12.
- **Non-seasonal differencing d:** chosen with the ADF test.
- **Order selection:** AIC over p, q ∈ {0, 1, 2} and P, Q ∈ {0, 1}, done **once on the first training window (2005–2013)** for each target.
- **Each forecast year:** parameters re-estimated (warm-started from the previous year), then 12 months forecast from the December origin.

In [3]:
# ============================================================
# 2. SARIMAX COMPONENTS (per grid, GHI and CSI)
# ============================================================

import ast
from itertools import product

from joblib import Parallel, delayed
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.stattools import adfuller


# ------------------------------------------------------------
# SARIMAX SETTINGS (same as the standalone model)
# ------------------------------------------------------------

SARIMAX_TARGETS = {
    "ghi": GHI_COL,
    "csi": CSI_COL
}

EXOG_LAG = 12

P_VALUES = [0, 1, 2]

Q_VALUES = [0, 1, 2]

SEASONAL_P_VALUES = [0, 1]

SEASONAL_Q_VALUES = [0, 1]

SEASONAL_D = 1

ADF_ALPHA = 0.05

MAX_ITER = 200

N_JOBS = -1


# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------

def fit_sarimax(endog, exog, order, seasonal_order, start_params=None):
    """Fit one SARIMAX model, silencing optimiser warnings."""

    with warnings.catch_warnings():

        warnings.simplefilter("ignore")

        return SARIMAX(
            endog,
            exog=exog,
            order=order,
            seasonal_order=seasonal_order
        ).fit(
            start_params=start_params,
            maxiter=MAX_ITER,
            disp=False
        )


def select_sarimax_order(endog_train, exog_train):
    """Choose d with the ADF test, then (p, q)(P, Q) by AIC."""

    seasonal_diff = endog_train.diff(SEASONAL_PERIOD).dropna()

    with warnings.catch_warnings():

        warnings.simplefilter("ignore", FutureWarning)

        adf_result = adfuller(seasonal_diff, autolag="AIC")

    adf_pvalue = adf_result[1] if isinstance(adf_result, tuple) else adf_result.pvalue

    d = 0 if adf_pvalue < ADF_ALPHA else 1

    best = None

    n_fitted = 0

    for p, q, P, Q in product(P_VALUES, Q_VALUES, SEASONAL_P_VALUES, SEASONAL_Q_VALUES):

        order = (p, d, q)

        seasonal_order = (P, SEASONAL_D, Q, SEASONAL_PERIOD)

        try:
            result = fit_sarimax(endog_train, exog_train, order, seasonal_order)
        except Exception:
            continue

        if not np.isfinite(result.aic):
            continue

        n_fitted += 1

        if best is None or result.aic < best[2].aic:
            best = (order, seasonal_order, result)

    if best is None:
        raise RuntimeError("No SARIMAX candidate could be fitted")

    return best[0], best[1], best[2], {"adf_pvalue": adf_pvalue, "candidates_fitted": n_fitted}


# ------------------------------------------------------------
# One grid: both targets, every forecast year
# ------------------------------------------------------------

def run_sarimax_grid(grid_number, frame, fixed_orders):
    """
    fixed_orders = None      -> select orders on the first training window
    fixed_orders = {target: (order, seasonal_order)} -> use them
    """

    series = frame.set_index("date").asfreq("MS")

    exog = series[EXOG_COLS].shift(EXOG_LAG).iloc[EXOG_LAG:]

    first_train_end = COMPONENT_ROUNDS["train_end_date"].iloc[0]

    forecasts, diagnostics, orders = [], [], []

    for target, column in SARIMAX_TARGETS.items():

        endog = series[column].iloc[EXOG_LAG:]

        result = None

        if fixed_orders is None:

            order, seasonal_order, result, selection = select_sarimax_order(
                endog.loc[:first_train_end],
                exog.loc[:first_train_end]
            )

            orders.append({
                "grid_number": grid_number,
                "target": target,
                "order": str(order),
                "seasonal_order": str(seasonal_order),
                "selection_window": f"{DATA_START[:7]} to {first_train_end:%Y-%m}",
                **selection
            })

        else:

            order, seasonal_order = fixed_orders[target]


        for position, rnd in enumerate(COMPONENT_ROUNDS.itertuples(index=False)):

            # The selected model already covers the first training window
            if not (fixed_orders is None and position == 0):

                result = fit_sarimax(
                    endog.loc[:rnd.train_end_date],
                    exog.loc[:rnd.train_end_date],
                    order,
                    seasonal_order,
                    start_params=None if result is None else result.params
                )

            forecast = result.get_forecast(
                steps=HORIZON,
                exog=exog.loc[rnd.forecast_start:rnd.forecast_end]
            ).predicted_mean

            burn_in = order[1] + SEASONAL_D * SEASONAL_PERIOD

            forecasts.append(pd.DataFrame({
                "component":   f"sarimax_{target}",
                "year":        rnd.year,
                "grid_number": grid_number,
                "target_date": forecast.index,
                "predicted":   forecast.to_numpy()
            }))

            diagnostics.append({
                "component":         f"sarimax_{target}",
                "year":              rnd.year,
                "grid_number":       grid_number,
                "order":             str(order),
                "seasonal_order":    str(seasonal_order),
                "aic":               result.aic,
                "converged":         bool(result.mle_retvals.get("converged", True)),
                "ljung_box_p_lag12": acorr_ljungbox(
                    result.resid.iloc[burn_in:], lags=[SEASONAL_PERIOD]
                )["lb_pvalue"].iloc[0]
            })

    return forecasts, diagnostics, orders


# ------------------------------------------------------------
# Orders: select (rolling) or load (final test)
# ------------------------------------------------------------

if STAGE == "rolling":

    FIXED_ORDERS = {grid_number: None for grid_number in GRID_INFO["grid_number"]}

else:

    if not SARIMAX_ORDERS_FILE.exists():
        raise FileNotFoundError(f"{SARIMAX_ORDERS_FILE} not found. Run Hybrid_Models.ipynb first.")

    order_table = pd.read_csv(SARIMAX_ORDERS_FILE)

    FIXED_ORDERS = {
        grid_number: {
            row["target"]: (ast.literal_eval(row["order"]), ast.literal_eval(row["seasonal_order"]))
            for _, row in rows.iterrows()
        }
        for grid_number, rows in order_table.groupby("grid_number")
    }


# ------------------------------------------------------------
# Run all grids in parallel
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("SARIMAX COMPONENTS (GHI and CSI)")
print("=" * 65)

sarimax_outputs = Parallel(n_jobs=N_JOBS, verbose=5)(
    delayed(run_sarimax_grid)(grid_number, frame, FIXED_ORDERS[grid_number])
    for grid_number, frame in data.groupby("grid_number")
)

sarimax_raw = pd.concat(
    [table for output in sarimax_outputs for table in output[0]],
    ignore_index=True
)

sarimax_diagnostics = pd.DataFrame(
    [row for output in sarimax_outputs for row in output[1]]
)

sarimax_diagnostics.to_csv(MODEL_DIR / f"sarimax_component_diagnostics_{STAGE}.csv", index=False)

if STAGE == "rolling":

    pd.DataFrame(
        [row for output in sarimax_outputs for row in output[2]]
    ).to_csv(SARIMAX_ORDERS_FILE, index=False)

    print(f"Orders saved: {SARIMAX_ORDERS_FILE}")

print(
    f"Fits not converged: {(~sarimax_diagnostics['converged']).sum()} "
    f"of {len(sarimax_diagnostics)}"
)


# ------------------------------------------------------------
# Component forecasts in both GHI and CSI space
# ------------------------------------------------------------

def finish_component(raw, component, target):
    """Express a component forecast as both GHI and CSI."""

    frame = raw.merge(
        data[["grid_number", "date", CLEAR_COL]].rename(columns={"date": "target_date"}),
        on=["grid_number", "target_date"],
        how="left"
    )

    if target == "csi":
        frame["predicted_csi"] = frame["predicted"]
        frame["predicted_ghi"] = frame["predicted"] * frame[CLEAR_COL]
    else:
        frame["predicted_ghi"] = frame["predicted"]
        frame["predicted_csi"] = frame["predicted"] / frame[CLEAR_COL]

    frame["component"] = component

    frame = frame[["component", "year", "grid_number", "target_date", "predicted_ghi", "predicted_csi"]]

    frame.to_csv(COMPONENT_DIR / f"{component}_{STAGE}.csv", index=False)

    return frame


components = {}

for target in SARIMAX_TARGETS:

    name = f"sarimax_{target}"

    components[name] = finish_component(
        sarimax_raw[sarimax_raw["component"] == name],
        name,
        target
    )

    print(f"Saved component: {COMPONENT_DIR / f'{name}_{STAGE}.csv'}")


SARIMAX COMPONENTS (GHI and CSI)


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 12 concurrent workers.
[Parallel(n_jobs=-1)]: Done  48 tasks      | elapsed: 31.5min
[Parallel(n_jobs=-1)]: Done  89 out of  93 | elapsed: 48.3min remaining:  2.2min
[Parallel(n_jobs=-1)]: Done  93 out of  93 | elapsed: 48.5min finished


Orders saved: outputs\hybrid_models\models\sarimax_component_orders.csv
Fits not converged: 22 of 1674
Saved component: outputs\hybrid_models\components\sarimax_ghi_rolling.csv
Saved component: outputs\hybrid_models\components\sarimax_csi_rolling.csv


## 3. LSTM components

These use the same global multi-output LSTM as the standalone model:
- **Architecture:** 2 × 64 LSTM units, dropout 0.2, a linear layer with 12 outputs.
- **Training:** Adam (learning rate 1e-3), batch size 64, early stopping (patience 20) on the last training year.
- **Input window:** 12 months ending at the December origin.
- **Samples:** every month is used as an origin for training; only the December origin is forecast.

There are three variants:

| Variant | Input series | Target | Loss | Used in |
|---|---|---|---|---|
| `lstm_ghi` | GHI, cloud amount, RH (scaled) | GHI (scaled) | MSE | stack 1 |
| `lstm_csi` | **CSI**, cloud amount, RH (scaled) | **CSI** | MSE | stack 2 |
| `lstm_csi_pl` | **CSI**, cloud amount, RH (scaled) | **CSI** | **penalized loss** | stack 3 |

All variants also receive sin/cos of the month and the grid's latitude, longitude and elevation. The CSI target is not scaled: CSI is already a dimensionless ratio of about 0.4–1, so the clear-sky threshold (CSI_max = 1) can be applied directly in the loss. Early stopping monitors each variant's own loss.

In [4]:
# ============================================================
# 3. LSTM COMPONENTS (global multi-output, PyTorch)
# ============================================================

import copy

import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import MinMaxScaler


# ------------------------------------------------------------
# LSTM SETTINGS (same as the standalone model)
# ------------------------------------------------------------

HIDDEN_SIZE = 64

NUM_LAYERS = 2

DROPOUT = 0.2

LEARNING_RATE = 1e-3

BATCH_SIZE = 64

MAX_EPOCHS = 200

PATIENCE = 20


# ------------------------------------------------------------
# PENALIZED LOSS SETTINGS (stack 3)
# ------------------------------------------------------------

PENALTY_SETTINGS = {
    "csi_max":          CSI_MAX,   # clear-sky threshold
    "tolerance":        0.05,      # acceptable CSI error (tau)
    "violation_weight": 10.0,      # lambda_v: clear-sky threshold violations
    "error_weight":     1.0,       # lambda_e: errors beyond the tolerance
    "reward_weight":    0.005      # lambda_r: reward for accurate, physical forecasts
}


# ------------------------------------------------------------
# VARIANTS
# ------------------------------------------------------------

LSTM_VARIANTS = {
    "lstm_ghi": {
        "inputs": [GHI_COL, CLOUD_COL, RH_COL],
        "target": GHI_COL,
        "scale_target": True,
        "loss": "mse"
    },
    "lstm_csi": {
        "inputs": [CSI_COL, CLOUD_COL, RH_COL],
        "target": CSI_COL,
        "scale_target": False,
        "loss": "mse"
    },
    "lstm_csi_pl": {
        "inputs": [CSI_COL, CLOUD_COL, RH_COL],
        "target": CSI_COL,
        "scale_target": False,
        "loss": "penalized"
    }
}

torch.use_deterministic_algorithms(True)


# ------------------------------------------------------------
# Penalized loss
# ------------------------------------------------------------

class PenalizedClearSkyLoss(nn.Module):
    """
    Physics-informed loss on the clear-sky index (CSI):

        L = MSE
          + violation_weight * mean[ReLU(pred - csi_max)^2 + ReLU(-pred)^2]
          + error_weight     * mean[ReLU(|pred - target| - tolerance)^2]
          - reward_weight    * mean[exp(-(pred - target)^2 / (2 tolerance^2))
                                    * 1(0 <= pred <= csi_max)]

    Penalties: forecasts that violate the clear-sky threshold
    (GHI above clear-sky GHI, or negative) and errors beyond the
    tolerance. Reward: bounded in [0, reward_weight], only for
    forecasts that are both accurate and physically valid.
    """

    def __init__(self, csi_max, tolerance, violation_weight, error_weight, reward_weight):

        super().__init__()

        self.csi_max = csi_max
        self.tolerance = tolerance
        self.violation_weight = violation_weight
        self.error_weight = error_weight
        self.reward_weight = reward_weight

    def forward(self, prediction, target):

        error = prediction - target

        mse = (error ** 2).mean()

        violation = (
            torch.relu(prediction - self.csi_max) ** 2
            + torch.relu(-prediction) ** 2
        ).mean()

        excess_error = (torch.relu(error.abs() - self.tolerance) ** 2).mean()

        physical = ((prediction >= 0) & (prediction <= self.csi_max)).float()

        reward = (
            torch.exp(-error ** 2 / (2 * self.tolerance ** 2)) * physical
        ).mean()

        return (
            mse
            + self.violation_weight * violation
            + self.error_weight * excess_error
            - self.reward_weight * reward
        )


def make_loss(kind):

    if kind == "penalized":
        return PenalizedClearSkyLoss(**PENALTY_SETTINGS)

    return nn.MSELoss()


# ------------------------------------------------------------
# Sequences: 12-month input window -> next 12 months
# ------------------------------------------------------------

def make_sequences(frame, feature_cols, target_col, window, horizon):
    """
    For every grid and origin month i:
        X = features of months i-window+1 ... i
        Y = target of months i+1 ... i+horizon
    """

    X, Y, grid_numbers, origins = [], [], [], []

    for grid_number, group in frame.groupby("grid_number"):

        features = group[feature_cols].to_numpy(dtype=np.float32)

        target = group[target_col].to_numpy(dtype=np.float32)

        dates = group["date"].to_numpy()

        for i in range(window - 1, len(group) - horizon):

            X.append(features[i - window + 1:i + 1])

            Y.append(target[i + 1:i + 1 + horizon])

            grid_numbers.append(grid_number)

            origins.append(dates[i])

    keys = pd.DataFrame({
        "grid_number": grid_numbers,
        "origin_date": pd.to_datetime(origins)
    })

    return np.stack(X), np.stack(Y), keys


# ------------------------------------------------------------
# Model
# ------------------------------------------------------------

class LSTMForecaster(nn.Module):
    """Stacked LSTM with a multi-output linear head."""

    def __init__(self, n_features, hidden_size, num_layers, dropout, horizon):

        super().__init__()

        self.lstm = nn.LSTM(
            input_size=n_features,
            hidden_size=hidden_size,
            num_layers=num_layers,
            dropout=dropout if num_layers > 1 else 0.0,
            batch_first=True
        )

        self.dropout = nn.Dropout(dropout)

        self.output = nn.Linear(hidden_size, horizon)

    def forward(self, x):

        sequence_output, _ = self.lstm(x)

        return self.output(self.dropout(sequence_output[:, -1, :]))


def train_lstm(X_fit, Y_fit, X_es, Y_es, loss_function):
    """Train with early stopping on the model's own loss."""

    set_seed(SEED)

    torch.manual_seed(SEED)

    model = LSTMForecaster(
        n_features=X_fit.shape[2],
        hidden_size=HIDDEN_SIZE,
        num_layers=NUM_LAYERS,
        dropout=DROPOUT,
        horizon=HORIZON
    )

    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

    loader = DataLoader(
        TensorDataset(torch.from_numpy(X_fit), torch.from_numpy(Y_fit)),
        batch_size=BATCH_SIZE,
        shuffle=True,
        generator=torch.Generator().manual_seed(SEED)
    )

    X_es = torch.from_numpy(X_es)

    Y_es = torch.from_numpy(Y_es)

    best_loss, best_state, best_epoch, wait = np.inf, None, 0, 0

    for epoch in range(1, MAX_EPOCHS + 1):

        model.train()

        for X_batch, Y_batch in loader:

            optimizer.zero_grad()

            loss = loss_function(model(X_batch), Y_batch)

            loss.backward()

            optimizer.step()

        model.eval()

        with torch.no_grad():
            es_loss = loss_function(model(X_es), Y_es).item()

        if es_loss < best_loss:

            best_loss, best_state, best_epoch, wait = es_loss, copy.deepcopy(model.state_dict()), epoch, 0

        else:

            wait += 1

            if wait >= PATIENCE:
                break

    model.load_state_dict(best_state)

    model.eval()

    return model, best_epoch, best_loss


# ------------------------------------------------------------
# Train every variant for every forecast year
# ------------------------------------------------------------

for variant, spec in LSTM_VARIANTS.items():

    print("\n" + "=" * 65)
    print(f"LSTM COMPONENT: {variant}  (target {spec['target']}, loss {spec['loss']})")
    print("=" * 65)

    scaled_cols = spec["inputs"] + STATIC_COLS

    feature_cols = scaled_cols + CALENDAR_COLS

    raw_forecasts = []

    for rnd in COMPONENT_ROUNDS.itertuples(index=False):

        frame = data[data["date"] <= rnd.forecast_end].copy()

        # Scaler fitted on this year's training rows only
        scaler = MinMaxScaler().fit(
            frame.loc[frame["date"] <= rnd.train_end_date, scaled_cols]
        )

        scaled = frame.copy()

        scaled[scaled_cols] = scaler.transform(frame[scaled_cols])

        scaled["__target__"] = scaled[spec["target"]] if spec["scale_target"] else frame[spec["target"]]

        X, Y, keys = make_sequences(scaled, feature_cols, "__target__", WINDOW, HORIZON)


        last_target = keys["origin_date"] + pd.DateOffset(months=HORIZON)

        early_stop_start = rnd.train_end_date - pd.DateOffset(months=11)

        fit_mask = (last_target < early_stop_start).to_numpy()

        es_mask = ((last_target >= early_stop_start) & (last_target <= rnd.train_end_date)).to_numpy()

        forecast_mask = (keys["origin_date"] == rnd.train_end_date).to_numpy()


        model, best_epoch, best_loss = train_lstm(
            X[fit_mask], Y[fit_mask],
            X[es_mask], Y[es_mask],
            make_loss(spec["loss"])
        )

        with torch.no_grad():
            predicted = model(torch.from_numpy(X[forecast_mask])).numpy()

        if spec["scale_target"]:

            index = scaled_cols.index(spec["target"])

            predicted = predicted * scaler.data_range_[index] + scaler.data_min_[index]


        grids = keys.loc[forecast_mask, "grid_number"].to_numpy()

        raw_forecasts.append(pd.DataFrame({
            "component":   variant,
            "year":        rnd.year,
            "grid_number": np.repeat(grids, HORIZON),
            "target_date": np.tile(pd.date_range(rnd.forecast_start, periods=HORIZON, freq="MS"), len(grids)),
            "predicted":   predicted.ravel()
        }))

        torch.save(
            {
                "state_dict": model.state_dict(),
                "variant": variant,
                "spec": spec,
                "penalty_settings": PENALTY_SETTINGS if spec["loss"] == "penalized" else None,
                "features": feature_cols,
                "scaled_cols": scaled_cols,
                "scaler_min": scaler.data_min_.tolist(),
                "scaler_max": scaler.data_max_.tolist(),
                "forecast_year": rnd.year,
                "best_epoch": best_epoch,
                "seed": SEED
            },
            MODEL_DIR / f"{variant}_{rnd.year}.pt"
        )

        print(
            f"Year {rnd.year}: fit {fit_mask.sum()}  early-stop {es_mask.sum()}  "
            f"best epoch {best_epoch}  early-stop loss {best_loss:.5f}"
        )

    components[variant] = finish_component(
        pd.concat(raw_forecasts, ignore_index=True),
        variant,
        "csi" if spec["target"] == CSI_COL else "ghi"
    )

    print(f"Saved component: {COMPONENT_DIR / f'{variant}_{STAGE}.csv'}")


LSTM COMPONENT: lstm_ghi  (target ghi_kwh_m2_day, loss mse)
Year 2014: fit 6789  early-stop 1116  best epoch 84  early-stop loss 0.00533
Year 2015: fit 7905  early-stop 1116  best epoch 35  early-stop loss 0.00565
Year 2016: fit 9021  early-stop 1116  best epoch 44  early-stop loss 0.00551
Year 2017: fit 10137  early-stop 1116  best epoch 29  early-stop loss 0.00803
Year 2018: fit 11253  early-stop 1116  best epoch 13  early-stop loss 0.00431
Year 2019: fit 12369  early-stop 1116  best epoch 36  early-stop loss 0.00512
Year 2020: fit 13485  early-stop 1116  best epoch 11  early-stop loss 0.00624
Year 2021: fit 14601  early-stop 1116  best epoch 21  early-stop loss 0.00500
Year 2022: fit 15717  early-stop 1116  best epoch 7  early-stop loss 0.00615
Saved component: outputs\hybrid_models\components\lstm_ghi_rolling.csv

LSTM COMPONENT: lstm_csi  (target csi, loss mse)
Year 2014: fit 6789  early-stop 1116  best epoch 102  early-stop loss 0.00306
Year 2015: fit 7905  early-stop 1116  best

## 4. Parallel ensemble: time-varying weighted average (TVWA)

For each stack, grid cell and forecast year Y:

1. Take the two components' out-of-sample GHI forecasts for **all earlier forecast years** and compute their discounted squared errors: `D_c = Σ_{y<Y} δ^(Y−1−y) Σ_months (GHI_c − GHI_obs)²`, with δ = 0.8.
2. Compute the SARIMAX weight: `w_S = D_LSTM / (D_SARIMAX + D_LSTM)`. The LSTM weight is `1 − w_S`. Both lie between 0 and 1.
3. Combine the forecasts: `GHI_hybrid = w_S · GHI_SARIMAX + (1 − w_S) · GHI_LSTM`.

The weights change from year to year (time-varying) and from grid to grid (spatially varying). They are saved in `models/<stack>_tvwa_weights.csv` so they can be analysed in the paper.

In [5]:
# ============================================================
# 4. PARALLEL ENSEMBLE: TIME-VARYING WEIGHTED AVERAGE
# ============================================================

# Discount factor: weight of an error made k+1 years before the forecast year = DISCOUNT^k
DISCOUNT = 0.8

HYBRIDS = {
    "hybrid": {
        "label": "Hybrid SARIMAX-LSTM",
        "sarimax": "sarimax_ghi",
        "lstm": "lstm_ghi"
    },
    "hybrid_csi": {
        "label": "Hybrid SARIMAX-LSTM + CSI",
        "sarimax": "sarimax_csi",
        "lstm": "lstm_csi"
    },
    "hybrid_csi_pl": {
        "label": "Hybrid SARIMAX-LSTM + CSI + PL",
        "sarimax": "sarimax_csi",
        "lstm": "lstm_csi_pl"
    }
}


# ------------------------------------------------------------
# Component forecasts, including the error history
# ------------------------------------------------------------

component_history = pd.concat(components.values(), ignore_index=True)

if STAGE == "final_test":

    # Earlier out-of-sample years from rolling validation
    rolling_files = [ROLLING_COMPONENT_DIR / f"{name}_rolling.csv" for name in components]

    missing = [str(path) for path in rolling_files if not path.exists()]

    if missing:
        raise FileNotFoundError(f"Run Hybrid_Models.ipynb first. Missing: {missing}")

    component_history = pd.concat(
        [pd.read_csv(path, parse_dates=["target_date"]) for path in rolling_files] + [component_history],
        ignore_index=True
    )

component_history = component_history.merge(
    data[["grid_number", "date", GHI_COL]].rename(columns={"date": "target_date", GHI_COL: "actual"}),
    on=["grid_number", "target_date"],
    how="left"
)

print(
    f"Component forecast years available for weighting: "
    f"{sorted(component_history['year'].unique())}"
)


# ------------------------------------------------------------
# TVWA
# ------------------------------------------------------------

def tvwa_combine(sarimax_name, lstm_name):
    """
    Combine two component forecasts with discounted inverse-MSE
    weights per grid, recomputed for every evaluation year.
    Returns (hybrid forecasts, weights table).
    """

    pair = (
        component_history[component_history["component"].isin([sarimax_name, lstm_name])]
        .pivot_table(
            index=["year", "grid_number", "target_date", "actual"],
            columns="component",
            values="predicted_ghi"
        )
        .reset_index()
        .rename(columns={sarimax_name: "sarimax", lstm_name: "lstm"})
    )

    forecasts, weights = [], []

    for year in EVAL_YEARS:

        history = pair[pair["year"] < year]

        if history.empty:
            raise ValueError(f"No earlier component forecasts to weight year {year}")

        discount = DISCOUNT ** (year - 1 - history["year"])

        discounted = (
            history.assign(
                d_sarimax=discount * (history["sarimax"] - history["actual"]) ** 2,
                d_lstm=discount * (history["lstm"] - history["actual"]) ** 2
            )
            .groupby("grid_number")[["d_sarimax", "d_lstm"]]
            .sum()
        )

        total = discounted["d_sarimax"] + discounted["d_lstm"]

        w_sarimax = (discounted["d_lstm"] / total).fillna(0.5)

        weights.append(pd.DataFrame({
            "year": year,
            "grid_number": w_sarimax.index,
            "w_sarimax": w_sarimax.to_numpy(),
            "w_lstm": 1 - w_sarimax.to_numpy(),
            "history_years": f"{history['year'].min()}-{history['year'].max()}"
        }))

        current = pair[pair["year"] == year].copy()

        current["w_sarimax"] = current["grid_number"].map(w_sarimax)

        current["predicted"] = (
            current["w_sarimax"] * current["sarimax"]
            + (1 - current["w_sarimax"]) * current["lstm"]
        )

        forecasts.append(current)

    return pd.concat(forecasts, ignore_index=True), pd.concat(weights, ignore_index=True)


hybrid_forecasts = {}

print("\nTVWA weights (mean SARIMAX weight across grids):")

for name, spec in HYBRIDS.items():

    forecasts, weights = tvwa_combine(spec["sarimax"], spec["lstm"])

    weights.to_csv(MODEL_DIR / f"{name}_tvwa_weights.csv", index=False)

    hybrid_forecasts[name] = forecasts

    print(
        f"  {spec['label']:<34} "
        + "  ".join(
            f"{year}: {w:.2f}"
            for year, w in weights.groupby("year")["w_sarimax"].mean().items()
        )
    )

Component forecast years available for weighting: [np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022)]

TVWA weights (mean SARIMAX weight across grids):
  Hybrid SARIMAX-LSTM                2015: 0.53  2016: 0.49  2017: 0.47  2018: 0.50  2019: 0.51  2020: 0.48  2021: 0.49  2022: 0.49
  Hybrid SARIMAX-LSTM + CSI          2015: 0.52  2016: 0.46  2017: 0.47  2018: 0.53  2019: 0.54  2020: 0.51  2021: 0.52  2022: 0.52
  Hybrid SARIMAX-LSTM + CSI + PL     2015: 0.54  2016: 0.50  2017: 0.51  2018: 0.52  2019: 0.53  2020: 0.51  2021: 0.51  2022: 0.51


## 5. Rolling-validation results

Each stack is scored on 2015–2022 with the same metrics and file format as the standalone models. Its components and climatology are scored too:
- **Components:** SARIMAX and LSTM in GHI and CSI form, needed for the ablation analysis.
- **Climatology:** the mean of each calendar month over the training years, used as the benchmark.

If `Standalone_Models.ipynb` has been run, its models are added to the summary table.

**Clear-sky violations** are forecasts with GHI above the clear-sky GHI (CSI > 1).

In [6]:
# ============================================================
# 5. ROLLING-VALIDATION RESULTS
# ============================================================

year_to_round = dict(zip(ROUNDS[YEAR_COL], ROUNDS["round"]))

EVALUATED = {}


def evaluate(forecasts, name, label):

    frame = forecasts[forecasts["year"].isin(EVAL_YEARS)].copy()

    frame["round"] = frame["year"].map(year_to_round)

    EVALUATED[label] = save_results(frame, name, label)


print("\n" + "=" * 65)
print(f"ROLLING VALIDATION {EVAL_YEARS[0]}-{EVAL_YEARS[-1]} (all rounds pooled)")
print("=" * 65)


# ------------------------------------------------------------
# Climatology benchmark
# ------------------------------------------------------------

climatology = []

for year in EVAL_YEARS:

    train = data[data["date"] < f"{year}-01-01"]

    targets = data.loc[data["year"] == year, ["grid_number", "date", "month"]]

    climatology.append(
        targets.merge(
            train.groupby(["grid_number", "month"], as_index=False)[GHI_COL].mean(),
            on=["grid_number", "month"]
        ).rename(columns={"date": "target_date", GHI_COL: "predicted"}).assign(year=year)
    )

evaluate(pd.concat(climatology, ignore_index=True), "climatology", "Climatology")


# ------------------------------------------------------------
# Components
# ------------------------------------------------------------

COMPONENT_LABELS = {
    "sarimax_ghi": "SARIMAX (GHI) component",
    "lstm_ghi":    "LSTM (GHI) component",
    "sarimax_csi": "SARIMAX (CSI) component",
    "lstm_csi":    "LSTM (CSI) component",
    "lstm_csi_pl": "LSTM (CSI + PL) component"
}

for name, label in COMPONENT_LABELS.items():

    evaluate(
        components[name].rename(columns={"predicted_ghi": "predicted"}),
        f"component_{name}",
        label
    )


# ------------------------------------------------------------
# Stacked hybrids
# ------------------------------------------------------------

for name, spec in HYBRIDS.items():

    evaluate(hybrid_forecasts[name], name, spec["label"])


# ------------------------------------------------------------
# Standalone models (from Standalone_Models.ipynb, if available)
# ------------------------------------------------------------

STANDALONE = {"sarimax": "SARIMAX", "lstm": "LSTM", "xgboost": "XGBoost"}

for name, label in STANDALONE.items():

    path = STANDALONE_PREDICTION_DIR / f"{name}_rolling_predictions.csv"

    if path.exists():

        frame = pd.read_csv(path, parse_dates=["target_date"]).merge(
            data[["grid_number", "date", CLEAR_COL]].rename(columns={"date": "target_date"}),
            on=["grid_number", "target_date"],
            how="left"
        )

        EVALUATED[f"{label} (standalone)"] = frame.assign(model=f"{label} (standalone)")


# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

all_predictions = pd.concat(EVALUATED.values(), ignore_index=True)

order = list(EVALUATED)

summary = summarise_errors(all_predictions, ["model"]).set_index("model").reindex(order)

round_rmse = (
    summarise_errors(all_predictions, ["model", YEAR_COL])
    .pivot(index=YEAR_COL, columns="model", values="RMSE")[order]
)

summary["round_RMSE_std"] = round_rmse.std()

summary["skill_vs_climatology"] = 1 - summary["RMSE"] / summary.loc["Climatology", "RMSE"]

summary["rounds_beating_climatology"] = round_rmse.lt(round_rmse["Climatology"], axis=0).sum()

summary["violation_pct"] = summary["clear_sky_violations"] / summary["n_months"] * 100

summary.to_csv(METRICS_DIR / "rolling_metrics_summary.csv")

pd.concat(
    [pd.read_csv(METRICS_DIR / f"{name}_rolling_metrics_by_grid.csv") for name in
     ["climatology"] + [f"component_{n}" for n in COMPONENT_LABELS] + list(HYBRIDS)],
    ignore_index=True
).to_csv(METRICS_DIR / "rolling_metrics_by_grid_all_models.csv", index=False)


print("\n" + "=" * 65)
print("SUMMARY")
print("=" * 65)

print("\nRMSE by validation year (kWh/m²/day), stacked hybrids vs climatology:")

print(round_rmse[["Climatology"] + [spec["label"] for spec in HYBRIDS.values()]].round(4).to_string())

print("\nAll rounds:")

print(
    summary[["RMSE", "MAE", "nRMSE_pct", "skill_vs_climatology",
             "rounds_beating_climatology", "clear_sky_violations", "violation_pct"]]
    .round(4)
    .to_string()
)

print(f"\nSaved: {METRICS_DIR / 'rolling_metrics_summary.csv'}")
print(f"Saved: {METRICS_DIR / 'rolling_metrics_by_grid_all_models.csv'}")


ROLLING VALIDATION 2015-2022 (all rounds pooled)
Climatology                        RMSE 0.4474   MAE 0.3574   nRMSE 8.64%   clear-sky violations 0
SARIMAX (GHI) component            RMSE 0.4579   MAE 0.3640   nRMSE 8.84%   clear-sky violations 1
LSTM (GHI) component               RMSE 0.4493   MAE 0.3577   nRMSE 8.67%   clear-sky violations 0
SARIMAX (CSI) component            RMSE 0.4566   MAE 0.3630   nRMSE 8.81%   clear-sky violations 0
LSTM (CSI) component               RMSE 0.4616   MAE 0.3608   nRMSE 8.91%   clear-sky violations 0
LSTM (CSI + PL) component          RMSE 0.4584   MAE 0.3602   nRMSE 8.85%   clear-sky violations 0
Hybrid SARIMAX-LSTM                RMSE 0.4342   MAE 0.3447   nRMSE 8.38%   clear-sky violations 0
Hybrid SARIMAX-LSTM + CSI          RMSE 0.4383   MAE 0.3471   nRMSE 8.46%   clear-sky violations 0
Hybrid SARIMAX-LSTM + CSI + PL     RMSE 0.4371   MAE 0.3456   nRMSE 8.44%   clear-sky violations 0

SUMMARY

RMSE by validation year (kWh/m²/day), stacked hyb

## References

The code was written for this study using the official APIs of the libraries below. It was not adapted from third-party code. Exact library versions are saved in `environment.txt` in the output folder.

**Software libraries**
- Anderson, K. S., Hansen, C. W., Holmgren, W. F., Jensen, A. R., Mikofski, M. A., & Driesse, A. (2023). pvlib python: 2023 project update. *Journal of Open Source Software, 8*(92), 5994. https://doi.org/10.21105/joss.05994. `Location.get_clearsky(model="ineichen")`: https://pvlib-python.readthedocs.io/en/stable/reference/generated/pvlib.clearsky.ineichen.html
- Seabold, S., & Perktold, J. (2010). Statsmodels: Econometric and statistical modeling with Python. *Proceedings of the 9th Python in Science Conference*, 92–96. https://doi.org/10.25080/Majora-92bf1922-011
- Paszke, A., Gross, S., Massa, F., Lerer, A., Bradbury, J., Chanan, G., … Chintala, S. (2019). PyTorch: An imperative style, high-performance deep learning library. *Advances in Neural Information Processing Systems 32*, 8024–8035. (custom loss via `torch.nn.Module`)
- Pedregosa, F., et al. (2011). Scikit-learn: Machine learning in Python. *Journal of Machine Learning Research, 12*, 2825–2830.
- Harris, C. R., et al. (2020). Array programming with NumPy. *Nature, 585*, 357–362. https://doi.org/10.1038/s41586-020-2649-2
- McKinney, W. (2010). Data structures for statistical computing in Python. *Proceedings of the 9th Python in Science Conference*, 56–61.

**Clear-sky model and clear-sky index**
- Ineichen, P., & Perez, R. (2002). A new airmass independent formulation for the Linke turbidity coefficient. *Solar Energy, 73*(3), 151–157. https://doi.org/10.1016/S0038-092X(02)00045-2
- Perez, R., Ineichen, P., Moore, K., Kmiecik, M., Chain, C., George, R., & Vignola, F. (2002). A new operational model for satellite-derived irradiances: Description and validation. *Solar Energy, 73*(5), 307–317. https://doi.org/10.1016/S0038-092X(02)00122-6
- Remund, J., Wald, L., Lefèvre, M., Ranchin, T., & Page, J. (2003). Worldwide Linke turbidity information. *Proceedings of ISES Solar World Congress 2003*, Göteborg, Sweden. (Linke turbidity climatology used by pvlib)
- Lauret, P., Voyant, C., Soubdhan, T., David, M., & Poggi, P. (2015). A benchmarking of machine learning techniques for solar radiation forecasting in an insular context. *Solar Energy, 112*, 446–457. https://doi.org/10.1016/j.solener.2014.12.014 (forecasting on the clear-sky index)

**Hybrid combination**
- Bates, J. M., & Granger, C. W. J. (1969). The combination of forecasts. *Journal of the Operational Research Society, 20*(4), 451–468. https://doi.org/10.1057/jors.1969.103 (inverse-MSE combination weights)
- Stock, J. H., & Watson, M. W. (2004). Combination forecasts of output growth in a seven-country data set. *Journal of Forecasting, 23*(6), 405–430. https://doi.org/10.1002/for.928 (discounted MSE, time-varying weights)
- Timmermann, A. (2006). Forecast combinations. In G. Elliott, C. W. J. Granger, & A. Timmermann (Eds.), *Handbook of Economic Forecasting* (Vol. 1, pp. 135–196). Elsevier. https://doi.org/10.1016/S1574-0706(05)01004-9

**Physics-informed / penalized loss**
- Karpatne, A., Watkins, W., Read, J., & Kumar, V. (2017). Physics-guided neural networks (PGNN): An application in lake temperature modeling. *arXiv:1710.11431*. https://arxiv.org/abs/1710.11431 (physics-based penalty terms in the loss)
- Raissi, M., Perdikaris, P., & Karniadakis, G. E. (2019). Physics-informed neural networks. *Journal of Computational Physics, 378*, 686–707. https://doi.org/10.1016/j.jcp.2018.10.045

**Models and evaluation** (see also `Standalone_Models.ipynb`)
- Box, G. E. P., Jenkins, G. M., Reinsel, G. C., & Ljung, G. M. (2015). *Time series analysis: Forecasting and control* (5th ed.). Wiley.
- Hochreiter, S., & Schmidhuber, J. (1997). Long short-term memory. *Neural Computation, 9*(8), 1735–1780. https://doi.org/10.1162/neco.1997.9.8.1735
- Tashman, L. J. (2000). Out-of-sample tests of forecasting accuracy: An analysis and review. *International Journal of Forecasting, 16*(4), 437–450. https://doi.org/10.1016/S0169-2070(00)00065-0